In [ ]:
나는 지금부터 이 IMU를 사용해서 코드를 짤거야. 너가 공부를 해놔. 

** 내시경이 현재 바라보는 방향을 기준으로 하여, 내시경헤드를 움직이고 나서, 얼마나 방향이 바뀌었는지** 를 출력하는 코드를 만들거야.  

자세변화를 측정해서 출력하는걸 할거야. 내 연구계획을 보여줄게. 
**내시경 카메라옆에 거리측정 센서(이건 별개의 센서입력이 있을 예정이야)를 달고, IMU의 x axis가 카메라와 같은방향으로 달아서**
**두 포인트와 내시경사이의 거리를 측정하고**
**두 포인트를 pointing할때 내시경의 헤드가 꺽인 정도를 IMU로 측정하는**
태스크를 할거야. 

지금은, 우선 거리는 출력하지 않고, **EBIMU를 이용한 자세**에 대해서만 코드를 짤거야. 

** 내시경이 현재 바라보는 방향을 기준으로 하여, 내시경헤드를 움직이고 나서, 얼마나 방향이 바뀌었는지** 를 출력하는 코드를 만들거야. 
그런데, 여기에서 어떤것을 고려해야 할까? 

내가말한 각도는, 벡터로 비유하자면, 
**초음 기준시점에 내시경이 바라보는 방향 벡터**
와
**현재 내시경이 바라보는 방향 벡터** 
의 사잇각을 나타내려고 하는것이야. 

또한, 코드에서 delta roll/pitch/yaw는 절대좌표계인지 상대좌표계인지도 파악해야해

우선, 학습을 너혼자서 하고 내가 이미 짠 코드를 기반으로, 컨펌받아야 할 것들을 생각해봐 

In [ ]:
#!pip install pyserial
#!pip install adafruit-blinka adafruit-circuitpython-vl53l4cd


In [2]:
## EBIMU가 어느 USB port에 연결되어있는지 보는 코드: CP2102N USB to UART Bridge Controller 가 EBIMU임. 
import serial.tools.list_ports
for p in serial.tools.list_ports.comports():
    print(p.device, "|", p.description)

## TOF I2C 실행 전 체크:
!i2cdetect -y 1    # 여기서 29가 보여야 합니다. VL53L4CD 기본 I2C 주소는 0x29입니다.

/dev/ttyS0 | n/a
/dev/ttyUSB0 | CP2102N USB to UART Bridge Controller


     0  1  2  3  4  5  6  7  8  9  a  b  c  d  e  f
00:                         -- -- -- -- -- -- -- -- 
10: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- 
20: -- -- -- -- -- -- -- -- -- 29 -- -- -- -- -- -- 
30: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- 
40: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- 
50: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- 
60: -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- -- 
70: -- -- -- -- -- -- -- --                         


In [2]:
## TOF값이 잘 나오는지 확인 

import time
import board
import adafruit_vl53l4cd
i2c = board.I2C()  # Raspberry Pi I2C 사용
tof = adafruit_vl53l4cd.VL53L4CD(i2c)
tof.start_ranging()
while True:
    if tof.data_ready:
        print(f"거리: {tof.distance} cm")
        tof.clear_interrupt()
    time.sleep(0.05)

거리: 46.7 cm
거리: 49.2 cm
거리: 46.4 cm
거리: 48.5 cm
거리: 46.6 cm
거리: 49.0 cm
거리: 46.5 cm
거리: 48.5 cm
거리: 46.4 cm
거리: 48.6 cm
거리: 46.8 cm
거리: 49.2 cm
거리: 47.5 cm
거리: 48.3 cm
거리: 46.8 cm
거리: 49.4 cm
거리: 46.6 cm
거리: 48.4 cm
거리: 46.4 cm
거리: 48.5 cm
거리: 46.7 cm
거리: 49.0 cm
거리: 46.4 cm
거리: 49.0 cm
거리: 46.8 cm
거리: 49.2 cm
거리: 46.8 cm
거리: 49.4 cm
거리: 47.0 cm
거리: 48.6 cm
거리: 47.2 cm
거리: 48.4 cm


KeyboardInterrupt: 

In [3]:
### 송신기(센서) 설정값 / 수신기 설정값 출력

import serial
import time

COM_PORT = "/dev/ttyUSB0"
BAUDRATE = 921600

ser = serial.Serial(COM_PORT, BAUDRATE, timeout=0.05)
time.sleep(0.2)

# 스트리밍 잠시 정지
ser.write(b"<stop>")
time.sleep(0.1)

# =========================
# 1) 수신기 cfg
# =========================
ser.reset_input_buffer()
ser.write(b"<cfg>")
time.sleep(0.2)

print("\n[RCV CFG START]")
t_end = time.time() + 1.0
while time.time() < t_end:
    n = ser.in_waiting
    if n > 0:
        print(ser.read(n).decode("utf-8", errors="ignore"), end="")
    time.sleep(0.02)
ser.write(b">")
print("\n[RCV CFG END]")

# =========================
# 2) 센서(ID=07) cfg
# =========================
ser.reset_input_buffer()
ser.write(b"<07cfg>")   # ID가 7일 때
time.sleep(0.2)

print("\n[SENSOR 07 CFG START]")
t_end = time.time() + 1.0
while time.time() < t_end:
    n = ser.in_waiting
    if n > 0:
        print(ser.read(n).decode("utf-8", errors="ignore"), end="")
    time.sleep(0.02)
ser.write(b">")
print("\n[SENSOR 07 CFG END]")

# 다시 시작
ser.write(b"<start>")
ser.write(b"<sof1>")

ser.close()


[RCV CFG START]
<
sb:8
sor:16
soc:1
sof:1
sog:0
soa:0
som:0
sod:2
sot:0
sob:1
sots:0
sch:100
mid:14
pons:1
>
[RCV CFG END]

[SENSOR 07 CFG START]
<ok><
sem:2
ssg:1
ssa:1
lpfg:3
lpfa:3
sffa:1
sffm:1
raa_l:0.050000
raa_t:1000
rha_l:0.075000
rha_t:10000
rha_fh:0
agc_e:1
agc_t:0.500000
agc_d:0.300000
cmo:0
posf_sl:0.000500
avca_e:0
avcg_e:0
sch:100
sid:7
sled:1
>
[SENSOR 07 CFG END]


In [ ]:
## EBIMU 데이터 출력 (테스트용)

import serial
import time
import math

# =========================
# 사용자 설정
# =========================
COM_PORT = "/dev/ttyUSB0"
BAUDRATE = 921600
PRINT_HZ = 20  # 출력 주기(Hz)

# =========================
# 유틸 함수
# =========================
def wrap_deg(a):
    """각도를 -180~180 범위로 정규화"""
    a = (a + 180.0) % 360.0 - 180.0
    return a


def parse_ebimu_euler_line(line):
    """
    예: '100-7,-0.01,-0.62,-53.40,95'
    반환: (roll, pitch, yaw) 또는 None
    """
    parts = line.split(",")
    if len(parts) < 4:
        return None

    # 첫 필드: CH-ID 형태인지 확인
    if "-" not in parts[0]:
        return None

    try:
        roll = float(parts[1])
        pitch = float(parts[2])
        yaw = float(parts[3])
        return (roll, pitch, yaw)
    except ValueError:
        return None


def forward_from_euler_deg(roll, pitch, yaw):
    """
    IMU x축이 카메라 시선축이라고 가정할 때,
    Euler(deg)에서 월드좌표계 시선 단위벡터를 계산.
    (ZYX: yaw-pitch-roll 기준)
    """
    p = math.radians(pitch)
    y = math.radians(yaw)

    # 바디 x축 [1,0,0]을 월드로 회전시킨 결과
    vx = math.cos(y) * math.cos(p)
    vy = math.sin(y) * math.cos(p)
    vz = -math.sin(p)

    norm = math.sqrt(vx * vx + vy * vy + vz * vz)
    if norm == 0.0:
        return (1.0, 0.0, 0.0)
    return (vx / norm, vy / norm, vz / norm)


def angle_between_vectors_deg(v0, v1):
    """두 단위벡터 사잇각(deg)"""
    dot = v0[0] * v1[0] + v0[1] * v1[1] + v0[2] * v1[2]
    dot = max(-1.0, min(1.0, dot))  # 수치 안정화
    return math.degrees(math.acos(dot))


# =========================
# 메인
# =========================
ser = serial.Serial(COM_PORT, BAUDRATE, timeout=1)
time.sleep(0.2)


def send_cmd(cmd):
    # EBRCV 명령 전송 (예: <sof1>)
    ser.write(cmd.encode("ascii"))


# Euler 출력 강제(권장), 배터리 출력은 있어도 파싱 가능
send_cmd("<sof1>")  # 자세를 Roll,Pitch,Yaw로 출력
time.sleep(0.1)

print("시리얼 연결됨.")
print("기준 방향으로 내시경 헤드를 고정한 뒤 Enter를 누르세요.")
input()

ref_rpy = None
ref_view_vec = None
last_print_t = 0.0

try:
    while True:
        raw = ser.read_until(b"\n")
        if not raw:
            continue

        line = raw.decode("utf-8", errors="ignore").strip()
        if not line:
            continue

        rpy = parse_ebimu_euler_line(line)
        if rpy is None:
            continue

        if ref_rpy is None:
            ref_rpy = rpy
            ref_view_vec = forward_from_euler_deg(*ref_rpy)
            print(f"[기준 설정] Roll={ref_rpy[0]:+.2f}, Pitch={ref_rpy[1]:+.2f}, Yaw={ref_rpy[2]:+.2f}")
            print("이제 내시경 헤드를 움직이면 기준 시선 대비 사잇각을 출력합니다.")
            continue

        now = time.time()
        if (now - last_print_t) < (1.0 / PRINT_HZ):
            continue
        last_print_t = now

        d_roll = wrap_deg(rpy[0] - ref_rpy[0])
        d_pitch = wrap_deg(rpy[1] - ref_rpy[1])
        d_yaw = wrap_deg(rpy[2] - ref_rpy[2])

        cur_view_vec = forward_from_euler_deg(*rpy)
        view_angle = angle_between_vectors_deg(ref_view_vec, cur_view_vec)

        print(
            f"ΔRoll={d_roll:+7.2f} deg | "
            f"ΔPitch={d_pitch:+7.2f} deg | "
            f"ΔYaw={d_yaw:+7.2f} deg | "
            f"시선벡터 사잇각={view_angle:6.2f} deg"
        )

except KeyboardInterrupt:
    print("\n종료합니다.")
finally:
    ser.close()
    print("포트 닫힘")

In [25]:
## IMU파라미터 및 캘리브레이션 등. (그런데, 가급적 터미널에서 하는게 좋을듯...)   ## EBIMU 데이터 출력 (테스트용)을 먼저 실행해야 한다. 

# 시작 시 1회 실행
send_cmd("<sof1>")     # Euler 출력
time.sleep(0.1)

send_cmd("<agc_e1>")   # (선택) Auto Gyro Calibration ON
time.sleep(0.1)

print("센서를 2초간 완전히 정지하세요")
send_cmd("<cg>")       # Gyro calibration
time.sleep(2.2)        # <ok> 대기 시간

센서를 2초간 완전히 정지하세요


In [26]:
## 본격코드(1/2) 준비 (커널 재시작 후 이 셀을 먼저 실행) - 함수생성임


import serial   # 시리얼(USB) 통신 라이브러리 (IMU와 대화할 때 사용)
import time     # 시간 지연(sleep) 등을 위한 라이브러리
import math     # 수학 함수(cos, sqrt 등) 라이브러리
import board                  # 라즈베리파이의 핀(I2C 등)을 다루는 라이브러리
import adafruit_vl53l4cd      # TOF 거리센서(VL53L4CD) 전용 라이브러리

# ---------- IMU(시리얼) 연결 ----------
COM_PORT = "/dev/ttyUSB0"     # IMU 수신기가 꽂힌 USB 포트 이름
BAUDRATE = 921600             # 통신 속도 (수신기 설정과 같아야 함)

ser = serial.Serial(COM_PORT, BAUDRATE, timeout=1)  # 포트를 열어서 ser라는 이름으로 사용
time.sleep(0.2)               # 포트가 안정될 때까지 0.2초 대기

# ---------- TOF(I2C) 연결 ----------
i2c = board.I2C()                       # 라즈베리파이의 I2C 통신 준비
tof = adafruit_vl53l4cd.VL53L4CD(i2c)   # TOF 센서를 tof라는 이름으로 사용
tof.start_ranging()                     # TOF 센서가 거리 측정을 시작하도록 명령

# ---------- IMU 데이터 해석용 함수들 ----------
def parse_ebimu_euler_line(line):
    # IMU가 보낸 한 줄(예: '100-7,0.32,0.29,153.40,99')에서
    # roll, pitch, yaw 세 각도를 뽑아내는 함수
    parts = line.split(",")          # 쉼표로 잘라서 리스트로 만듦
    if len(parts) < 4:               # 조각이 4개보다 적으면 (불완전한 줄)
        return None                  # "실패" 표시로 None을 돌려줌
    if "-" not in parts[0]:          # 첫 조각에 '-'가 없으면 (채널-ID 형식이 아니면)
        return None                  # 역시 실패 처리
    try:
        roll = float(parts[1])       # 두 번째 조각을 숫자로 변환 → roll
        pitch = float(parts[2])      # 세 번째 조각 → pitch
        yaw = float(parts[3])        # 네 번째 조각 → yaw
        return (roll, pitch, yaw)    # 세 각도를 묶어서 돌려줌
    except ValueError:               # 숫자 변환에 실패하면 (깨진 데이터)
        return None                  # 실패 처리

def forward_from_euler_deg(roll, pitch, yaw):
    # roll/pitch/yaw 각도로부터 "내시경이 바라보는 방향"을
    # 3차원 화살표(벡터)로 계산하는 함수
    p = math.radians(pitch)          # pitch를 라디안 단위로 변환
    y = math.radians(yaw)            # yaw를 라디안 단위로 변환
    vx = math.cos(y) * math.cos(p)   # 화살표의 x성분
    vy = math.sin(y) * math.cos(p)   # 화살표의 y성분
    vz = -math.sin(p)                # 화살표의 z성분
    return (vx, vy, vz)              # 길이 1짜리 방향 화살표 완성

def angle_between_vectors_deg(v0, v1):
    # 두 방향 화살표 사이의 각도(도 단위)를 구하는 함수
    dot = v0[0]*v1[0] + v0[1]*v1[1] + v0[2]*v1[2]  # 내적(두 화살표가 얼마나 같은 방향인지)
    dot = max(-1.0, min(1.0, dot))   # 계산 오차로 1을 살짝 넘는 것을 방지
    return math.degrees(math.acos(dot))  # 내적 → 각도로 변환해서 돌려줌

print("준비 완료")   # 여기까지 에러 없이 오면 준비 성공

준비 완료


In [27]:
## 본격코드(2/2) 실행 (기준점 → 병변 → 거리 계산) - 실행하는것임

OFFSET_CM = 2.3   # 센서와 내시경 회전중심 사이 거리 23mm = 2.3cm (측정값에 더해줌)

# ========== 1단계: 기준점 측정 ==========
print("1) 기준점을 조준하고 Enter를 누르세요")
input()                              # 사용자가 Enter를 누를 때까지 기다림

ser.reset_input_buffer()             # 버퍼에 쌓인 "과거" IMU 데이터를 모두 버림 (중요!)
v1 = None                            # 기준점 방향 화살표를 담을 변수 (아직 없음)
while v1 is None:                    # 유효한 데이터를 얻을 때까지 반복
    raw = ser.read_until(b"\n")      # IMU에서 한 줄 읽기 (줄바꿈까지)
    line = raw.decode("utf-8", errors="ignore").strip()  # 바이트를 글자로 바꾸고 공백 제거
    rpy = parse_ebimu_euler_line(line)   # 그 줄에서 roll/pitch/yaw 추출 시도
    if rpy is not None:                  # 추출에 성공했으면
        v1 = forward_from_euler_deg(*rpy)  # 방향 화살표로 변환해서 v1에 저장

while not tof.data_ready:            # TOF 센서에 새 측정값이 준비될 때까지 기다림
    time.sleep(0.01)                 # 0.01초씩 쉬면서 기다림
a = tof.distance + OFFSET_CM         # 거리(cm)에 회전중심 보정 2.3cm를 더해 a에 저장
tof.clear_interrupt()                # "값 읽었음"을 센서에 알림 (다음 측정 준비)

print(f"   기준점 거리 a = {a:.1f} cm")   # 소수점 1자리로 a 출력

# ========== 2단계: 병변 측정 ==========
print("2) 병변을 조준하고 Enter를 누르세요")
input()                              # 다시 Enter 대기

ser.reset_input_buffer()             # 이번에도 과거 데이터를 버리고
v2 = None                            # 병변 방향 화살표 변수
while v2 is None:                    # 유효한 데이터가 나올 때까지 반복
    raw = ser.read_until(b"\n")      # 한 줄 읽기
    line = raw.decode("utf-8", errors="ignore").strip()  # 글자로 변환
    rpy = parse_ebimu_euler_line(line)   # 각도 추출
    if rpy is not None:                  # 성공하면
        v2 = forward_from_euler_deg(*rpy)  # 방향 화살표 v2 저장

while not tof.data_ready:            # TOF 새 값 대기
    time.sleep(0.01)
b = tof.distance + OFFSET_CM         # 병변까지 거리 + 보정 → b
tof.clear_interrupt()                # 읽음 처리

print(f"   병변 거리 b = {b:.1f} cm")

# ========== 3단계: 코사인법칙으로 거리 계산 ==========
theta = angle_between_vectors_deg(v1, v2)   # 두 화살표 사이 각도 (사잇각, 도 단위)

# 제2코사인법칙: c² = a² + b² - 2ab·cos(θ)
c = math.sqrt(a*a + b*b - 2*a*b*math.cos(math.radians(theta)))

print(f"\n사잇각 θ = {theta:.2f} deg")        # 사잇각 출력
print(f"기준점-병변 거리 c = {c:.2f} cm")      # 최종 결과 출력

1) 기준점을 조준하고 Enter를 누르세요
   기준점 거리 a = 16.6 cm
2) 병변을 조준하고 Enter를 누르세요
   병변 거리 b = 16.6 cm

사잇각 θ = 45.57 deg
기준점-병변 거리 c = 12.86 cm
